# Compiling post-AGB stars table

From Oomen et al. 2018
 https://ui.adsabs.harvard.edu/abs/2018A%26A...620A..85O/abstract

In [1]:
import numpy as np
import pandas as pd
import h5py
import json
import subprocess, re, ast

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

## Latex Tables 
(copy pasted directly from ArXiv > Other Formats > Download Source)

From Oomen et al. 2018
 https://ui.adsabs.harvard.edu/abs/2018A%26A...620A..85O/abstract


## Table 1 data :

``` latex

\subsection{Orbital fitting}
\begin{table*}[]
\centering
\caption{Updated orbital elements}
\label{tableorbitalelements}
\begin{tabular}{ll|ccccccc} 
\hline
\hline
$\#$ & Star name & Period (days) & Eccentricity & T$_0$ (days) & $\omega$ $(^{\circ})$ & K$_1$ (km/s) & $\gamma$ (km/s) & ref. \\
\hline
1  & 89~Her         & 289.1$\pm$0.2    & 0.29$\pm$0.07    & 2447832$\pm$12   & 68.4$\pm$15  & 4.2$\pm$0.3    & -27.0$\pm$0.2  &  1 \\
2  & AC~Her         & 1188.9$\pm$1.2   & 0.0$+$0.05    & /         & /    & 10.8$\pm$0.7   & -28.8$\pm$0.5  &  2\\
3  & BD+39~4926     & 871.7$\pm$0.4    & 0.024$\pm$0.006 & 2451040$\pm$36  & 150.8$\pm$15 & 16.06$\pm$0.08 & -30.45$\pm$0.07 & 3 \\
4  & BD+46~442      & 140.82$\pm$0.02  & 0.085$\pm$0.005 & 2455233.9$\pm$1.3   & 275.8$\pm$3.3  & 23.8$\pm$0.1   & -98.13$\pm$0.08 & 4 \\
5  & DY~Ori         & 1248$\pm$36    & 0.22$\pm$0.08   & 2455990$\pm$56  & 87.2$\pm$19  & 12.4$\pm$1.0   & -0.11$\pm$0.04 &  5 \\
6  & EP~Lyr         & 1151$\pm$14    & 0.39$\pm$0.09   & 2455029.3$\pm$8.7  & 61.1$\pm$7.6  & 13.4$\pm$1.3   & 15.9$\pm$1.0   &  5\\
7  & HD~44179        & 317.6$\pm$1.1    & 0.27$\pm$0.03   & 2408735.9$\pm$5.3   & 346.6$\pm$4.8  & 12.1$\pm$0.3   & 20.0$\pm$0.2   &  6, 7\\
8  & HD~46703        & 597.4$\pm$0.2    & 0.30$\pm$0.02   & 2443519.6$\pm$7.6   & 241.9$\pm$4.2   & 16.0$\pm$0.3   & -93.3$\pm$0.2  & 8 \\
9  & HD~52961        & 1288.6$\pm$0.3  & 0.23$\pm$0.01   & 2407308$\pm$24  & 297.4$\pm$5.9  & 13.1$\pm$0.3   & 6.2$\pm$0.2     &   7 \\
10 & HD~95767        & 1989$\pm$61   & 0.25$\pm$0.05   & 2449500$\pm$95 & 197.7$\pm$19 & 12.1$\pm$0.8   & -20.3$\pm$0.8   & 9 \\
11 & HD~108015       & 906.3$\pm$5.9    & 0.0$+$0.03    & /         & /    & 3.4$\pm$0.3    & 4.0$\pm$0.2     & 9 \\
12 & HD~131356       & 1488.0$\pm$8.7   & 0.32$\pm$0.04   & 2449398$\pm$32  & 162.7$\pm$6.9  & 16.3$\pm$0.7   & -6.7$\pm$0.4    & 9 \\
13 & HD~158616       & 363.3$\pm$1.0    & 0.0$+$0.1     & /         & /    & 8.4$\pm$1.0    & 56.1$\pm$0.7    & 10 \\
14 & HD~213985       & 259.6$\pm$0.7    & 0.21$\pm$0.05    & 2407110$\pm$16   & 104.8$\pm$26  & 31.4$\pm$1.0   & -42.0$\pm$0.9  & 9 \\
15 & HP~Lyr         & 1818$\pm$80    & 0.20$\pm$0.04    & 2456175$\pm$61  & 14.2$\pm$13  & 7.8$\pm$0.2    & -115.6$\pm$0.2 & 5 \\
16 & HR~4049         & 430.6$\pm$0.1    & 0.30$\pm$0.01   & 2447176.6$\pm$3.8   & 236.5$\pm$3.5  & 16.6$\pm$0.2   & -31.9$\pm$0.2  & 11 \\
17 & IRAS~05208-2035 & 234.38$\pm$0.04    & 0.0$+$0.02    & /         & /    & 18.4$\pm$0.2   & 35.6$\pm$0.1    & 17 \\
18 & IRAS~06165+3158 & 262.6$\pm$0.7    & 0.0$+$0.05    & /  		  & /  & 15.5$\pm$0.5   & -16.4$\pm$0.3   & \\
19 & IRAS~06452-3456 & 215.4$\pm$0.4    & 0.0$+$0.03    & /         & /    & 36.9$\pm$0.6   & 45.9$\pm$0.3 &    \\
20 & IRAS~08544-4431 & 501.1$\pm$1.0    & 0.20$\pm$0.02   & 2451499.6$\pm$7.8  & 230.1$\pm$5.9  & 8.8$\pm$0.2    & 62.4$\pm$0.1 &  12  \\
21 & IRAS~09144-4933 & 1762$\pm$27    & 0.30$\pm$0.04    & 2451302$\pm$39  & 145.7$\pm$7.7  & 14.5$\pm$0.6   & 29.6$\pm$0.5  &  5 \\
22 & IRAS~15469-5311 & 390.2$\pm$0.7    & 0.08$\pm$0.02   & 2451530$\pm$13  & 114.0$\pm$16 & 12.3$\pm$0.4   & -13.9$\pm$0.3  & 12 \\
23 & IRAS~16230-3410 & 649.8$\pm$3.5    & 0.0$+$0.13    & /         & /    & 3.9$\pm$0.3    & -154.3$\pm$0.2 & \\
24 & IRAS~17038-4815 & 1394$\pm$12    & 0.63$\pm$0.06   & 2451694$\pm$15  & 124.5$\pm$5.5  & 15.2$\pm$1.5   & -25.6$\pm$0.5  &  5 \\
25 & IRAS~19125+0343 & 519.7$\pm$0.7    & 0.24$\pm$0.03   & 2451503$\pm$11  & 243.0$\pm$8.1  & 12.0$\pm$0.5   & 67.3$\pm$0.3 &  12  \\
26 & IRAS~19135+3937 & 126.97$\pm$0.08  & 0.13$\pm$0.03   & 2454997.7$\pm$1.0   & 66.0$\pm$4.4   & 18.0$\pm$0.6   & 2.1$\pm$0.4 &  13  \\
27 & IRAS~19157-0247 & 119.6$\pm$0.1    & 0.34$\pm$0.04   & 2451366.5$\pm$2.9   & 72.1$\pm$8.0  & 8.0$\pm$0.4    & 31.7$\pm$0.3  & 12 \\
28 & RU~Cen         & 1489$\pm$10    & 0.62$\pm$0.07   & 2449885$\pm$25  & 315.2$\pm$12 & 22.1$\pm$1.9   & -25.9$\pm$0.8  & 14 \\
29 & SAO~173329      & 115.951$\pm$0.002  & 0.0$+$0.04   & /       & /    & 12.4$\pm$0.3   & 73.3$\pm$0.2    & 9 \\
30 & ST~Pup         & 406.0$\pm$2.2    & 0.0$+$0.04    & /         & /    & 17.9$\pm$0.7   & 0.1$\pm$0.2    & 15 \\
31 & SX~Cen         & 564.3$\pm$7.6    & 0.0$+$0.06    & /         & /    & 21.5$\pm$0.9   & 24.3$\pm$1.0   & 14 \\
32 & TW~Cam         & 662.2$\pm$5.3    & 0.25$\pm$0.04   & 2455111$\pm$18  & 144.4$\pm$10 & 14.1$\pm$0.6   & -49.8$\pm$0.5  & 5 \\
33 & U~Mon          & 2550$\pm$143   & 0.25$\pm$0.06    & 2451988$\pm$316 & 87.3$\pm$15 & 14.9$\pm$1.1   & 24.1$\pm$1.0   & 16 \\
\hline
\end{tabular}
\tablefoot{References point to previously published orbits of stars for which additional RV data was collected in order to update the available orbital elements.}
\tablebib{(1)~\citet{waters93}; (2)~\citet{vanwinckel98}; (3)~\citet{kodaira70}; (4)~\citet{gorlova12}; (5)~\citet{manick17}; (6)~\citet{waelkens96}; (7)~\citet{vanwinckel95}; (8)~\citet{hrivnak08}; (9)~\citet{vanwinckel00}; (10)~\citet{desmedt16}; (11)~\citet{waelkens91};  (12)~\citet{vanwinckel09}; (13)~\citet{gorlova15}; (14)~\citet{maas02}; (15)~\citet{gonzalez96};  (16)~\citet{pollard06}; (17)~\citet{gielen08}}
\end{table*}  

```


## Table 2 data

``` latex
\begin{table*}[] 
\centering
\caption{Projected semi-major axis, mass functions, and minimum masses}
\label{massfunctions}
\begin{tabular}{llccc}
\hline
\hline
\multirow{2}{*}{$\#$} & \multirow{2}{*}{Star name} & \multirow{2}{*}{$a_1 \sin i$ (AU)} & \multicolumn{1}{p{2cm}}{\centering Mass function\\($M_\sun$)} & \multicolumn{1}{p{3cm}}{\centering Minimum mass ($M_\sun$)\\($M_1 = 0.6$, $i=75^\circ$)} \\
\hline
1  & 89~Her         & 0.106$\pm$0.007   & 0.0019$\pm$0.0004 & 0.10\\
2  & AC~Her         & 1.176$\pm$0.080     & 0.153$\pm$0.032 & 0.64  \\
3  & BD+39~4926     & 1.286$\pm$0.007   & 0.373$\pm$0.006 & 1.03  \\
4  & BD+46~442      & 0.3074$\pm$0.0014 & 0.195$\pm$0.003 & 0.72  \\
5  & DY~Ori         & 1.39$\pm$0.11     & 0.23$\pm$0.05  & 0.79    \\
6  & EP~Lyr         & 1.30$\pm$0.12      & 0.22$\pm$0.06  & 0.77    \\
7  & HD~44179        & 0.342$\pm$0.008   & 0.053$\pm$0.003 & 0.38  \\
8  & HD~46703        & 0.839$\pm$0.015   & 0.220$\pm$0.012 & 0.77  \\
9  & HD~52961        & 1.507$\pm$0.034     & 0.274$\pm$0.019 & 0.87  \\
10 & HD~95767        & 2.14$\pm$0.16     & 0.33$\pm$0.07 & 0.96     \\
11 & HD~108015       & 0.28$\pm$0.02     & 0.0036$\pm$0.0009 & 0.13\\
12 & HD~131356       & 2.11$\pm$0.09     & 0.57$\pm$0.07  & 1.33     \\
13 & HD~158616       & 0.28$\pm$0.03     & 0.022$\pm$0.008 & 0.26  \\
14 & HD~213985       & 0.733$\pm$0.025   & 0.777$\pm$0.079 & 1.62  \\
15 & HP~Lyr         & 1.27$\pm$0.06     & 0.083$\pm$0.007 & 0.47  \\
16 & HR~4049         & 0.627$\pm$0.010   & 0.177$\pm$0.008 & 0.69    \\
17 & IRAS~05208-2035 & 0.396$\pm$0.004   & 0.150$\pm$0.005 & 0.64   \\
18 & IRAS~06165+3158 & 0.374$\pm$0.011   & 0.10$\pm$0.01  & 0.52      \\
19 & IRAS~06452-3456 & 0.73$\pm$0.01     & 1.12$\pm$0.05  & 2.07     \\
20 & IRAS~08544-4431 & 0.398$\pm$0.008     & 0.033$\pm$0.002 & 0.31    \\
21 & IRAS~09144-4933 & 2.25$\pm$0.11     & 0.49$\pm$0.07  & 1.21     \\
22 & IRAS~15469-5311 & 0.438$\pm$0.015   & 0.074$\pm$0.008  & 0.45 \\
23 & IRAS~16230-3410 & 0.232$\pm$0.021    & 0.004$\pm$0.001  & 0.13\\
24 & IRAS~17038-4815 & 1.52$\pm$0.08     & 0.24$\pm$0.04  & 0.81     \\
25 & IRAS~19125+0343 & 0.56$\pm$0.02     & 0.086$\pm$0.010 & 0.48  \\
26 & IRAS~19135+3937 & 0.209$\pm$0.008   & 0.075$\pm$0.008 & 0.45  \\
27 & IRAS~19157-0247 & 0.083$\pm$0.003   & 0.0053$\pm$0.0007 & 0.15\\
28 & RU~Cen         & 2.38$\pm$0.15     & 0.81$\pm$0.17  & 1.66    \\
29 & SAO~173329      & 0.132$\pm$0.003   & 0.023$\pm$0.001 & 0.27\\
30 & ST~Pup         & 0.67$\pm$0.02     & 0.241$\pm$0.026  & 0.81 \\
31 & SX~Cen         & 1.12$\pm$0.05     & 0.58$\pm$0.07  & 1.35    \\
32 & TW~Cam         & 0.83$\pm$0.04     & 0.174$\pm$0.022  & 0.68 \\
33 & U~Mon          & 3.38$\pm$0.31       & 0.79$\pm$0.18   & 1.64  \\
\hline
\end{tabular}
\end{table*}
```

## Table 3


``` latex

\begin{table*}[]
\centering
\caption{Spectroscopic data and results of SED fitting for post-AGB stars in the sample.}
\label{specdata}
\begin{tabular}{llcccccccccc}
\hline
\hline
$\#$ & Star name & $T_\mathrm{eff}$ (K) & $\log g$ & E(B-V)  &  $L_\mathrm{IR}/L_*$ & [Fe/H] & [Zn/Fe] & [Zn/Ti] & [S/Ti] & Depletion & Ref.\\
\hline
1  & 89~Her        & 6600 & 0.8   & 0.02 & 0.38  & -0.5  & 0.1   & 0.6   & 0.7 & mild & 1  \\
2  & AC~Her        & 5800 & 1.0   & 0.46 & 0.24  & -1.4  & 0.5   & 0.7   & 1.2 & mild & 2  \\
3  & BD+39~4926    & 7750 & 1.0   & 0.23 & 0.0   & -2.4  & 1.7   & 2.0   & 3.2 & strong & 3  \\
4  & BD+46~442     & 6250 & 1.5   & 0.23 & 0.19  & -0.8 & -0.1 & -0.2   & -0.4 & no & 4 \\
5  & DY~Ori        & 5900 & 1.5   & 0.90 & 0.74  & -2.3  & 2.1   & 2.1   & 2.5 & strong & 5  \\
6  & EP~Lyr        & 6200 & 1.5   & 0.48 & 0.04  & -1.8  & 1.1   & 1.3   & 1.4 & moderate & 5  \\
7  & HD~44179        & 7500 & 0.8   & 0.15 & 18.1  & -3.3  & 2.7   & / & / & strong & 6, 7 \\
8  & HD~46703        & 6250 & 1.0   & 0.23 & 0.02  & -1.7  & 0.8   & 0.9   & 1.1 & mild & 8  \\
9  & HD~52961        & 6000 & 0.5   & 0.04 & 0.13  & -4.8  & 3.4   & 3.0   & 3.4 & strong & 9, 10 \\
10 & HD~95767        & 7500 & 2.0   & 0.58 & 0.55  & 0.1   & -0.2  & 0.0   & 0.1 & no & 11  \\
11 & HD~108015       & 7000 & 1.5   & 0.15 & 1.04  & -0.1 & -0.1 & 0.1   & 0.1 & no & 11 \\
12 & HD~131356       & 6000 & 1.0   & 0.15 & 0.65  & 0.0   & 0.2   & 0.6  & 0.5 & mild & 11  \\
13 & HD~158616       & 7250 & 1.25   & 0.51 & 0.23  & -0.6  & 0.2   & 0.0   & 0.1 & no & 12  \\
14 & HD~213985       & 8250 & 1.5   & 0.12 & 0.35  & -0.9  & / & / & 1.9 & strong & 13  \\
15 & HP~Lyr        & 6300 & 1.0   & 0.39 & 0.56   & -1.0  & 0.6  & 2.6  & 3.0 & strong &  14 \\
16 & HR~4049         & 7600 & 1.1    & 0.20 & 0.12  & -4.8  & 3.5   & / & / & strong & 6 \\
17 & IRAS~05208-2035 & 4250 & 0.75    & 0.01 & 0.43 & -0.7 & / & / & / & no & 3 \\
18 & IRAS~06165+3158 & 4250 & 1.5 & 0.53 & 0.39 & -0.9 & -0.1 & 0.0 & / & no & 10 \\
19 & IRAS~06452-3456 & /  & /   & 0.94 &  0.11  & / & / & / & / & / & /\\
20 & IRAS~08544-4431 & 7250 & 1.5   & 1.32  & 0.49  & -0.3  & 0.4   & 0.9   & 1.0 & mild & 15  \\
21 & IRAS~09144-4933 & 5750 & 0.5   & 1.78  & 0.81  & -0.3  & / & / & 1.3 & moderate & 15  \\
22 & IRAS~15469-5311 & 7500 & 1.5   & 1.27 & 0.74   & 0.0   & 0.3   & 1.8   & 2.1  & strong & 15  \\
23 & IRAS~16230-3410 & 6250 & 1.0     & 0.72 & 0.46   & -0.7  & 0.3   & 1.0   & 1.1 & moderate & 15  \\
24 & IRAS~17038-4815 & 4750 & 0.5 & 0.57 & 0.79   & -1.5  & 0.3   & / & / & no & 15 \\
25 & IRAS~19125+0343 & 7750 & 1.0   & 0.94 & 0.90  & -0.3  & 0.4   & 2.3   & 2.6 & strong & 15  \\
26 & IRAS~19135+3937 & 6000 & 0.5   & 0.28 & 0.26  & -1.0 & 0.0 & / & / & no & 10\\
27 & IRAS~19157-0247 & 7750 & 1.0    & 0.66 & 0.79  & 0.1   & / & / & 0.4 & no & 15  \\
28 & RU~Cen        & 6000 & 1.5   & 0.18 & 0.39    & -1.9  & 0.9   & 1.0   & 1.3 & moderate & 16  \\
29 & SAO~173329      & 7000 & 1.0   & 0.31 & 0.35  & -0.9 & 0.1  & -0.1 & 0.4 & no & 3 \\
30 & ST~Pup        & 5500 & 1.0   & 0.06 & 1.32 & -1.5  & 1.4   & 2.1   & 2.0 & strong & 17  \\
31 & SX~Cen        & 6250 & 1.5   & 0.17 & 0.40  & -1.1  & 0.6   & 1.5   & 1.9 & strong & 16  \\
32 & TW~Cam        & 4800 & 0.0  & 0.42 & 0.43  & -0.5  & 0.1   & 0.3   & 0.7 & no & 18  \\
33 & U~Mon         & 5000 & 0.0   & 0.34 & 0.28   & -0.8  & 0.2   & 0.0   & 0.5 & no & 18 \\
\hline
\end{tabular}
\tablefoot{
Values for metallicity and surface gravity are in dex. Uncertainties are not quoted, but formal errors for temperature are $\pm$ 250~K, for surface gravity $\pm$ 0.5~dex, and for abundances $\pm$ 0.3~dex. Effective temperature, surface gravity, and abundances come from literature, quoted in the rightmost column. The E(B-V) values come from SED fitting (see Sect. \ref{sec:sedfitting}), and the infrared luminosities are computed by integrating the infrared excess in SEDs in Appendix~\ref{appendix:seds}.}
\tablebib{
(1)~\citet{kipper11}; (2) \citet{giridhar98}; (3) \citet{rao12}; (4) \citet{gorlova12}; (5) \citet{gonzalez97}; (6) \citet{vanwinckel95}; (7) \citet{waelkens96}; (8) \citet{hrivnak08}; (9) \citet{waelkens91}; (10) \citet{rao14}; (11) \citet{vanwinckel97}; (12) \citet{desmedt16}; (13) \citet{deruyter06}; (14) \citet{giridhar05}; (15) \citet{maas05}; (16) \citet{maas02}; (17) \citet{gonzalez96}; (18) \citet{giridhar00}.}
\end{table*}

```

## Build the h5 table

In [2]:
# ---------- scheme from data_table.ipynb ----------
columns = [
    "System Name", "RA", "Dec", "Period", "Eccentricity",
    "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",
    "Type1", "Type2", "Detection Method", "Reference", "Notes"
]
df = pd.DataFrame(columns=columns)

def add_observation(df, system_name,
                    ra, dec, period, ecc,
                    m1, m1_sin3i, m2, m2_sin3i, q, mass_func,
                    type1, type2, method, reference, notes=""):
    new_row = {
        "System Name": system_name,
        "RA": ra,
        "Dec": dec,
        "Period": period,
        "Eccentricity": ecc,
        "M1": m1,
        "M1_sin3i": m1_sin3i,
        "M2": m2,
        "M2_sin3i": m2_sin3i,
        "q": q,
        "Mass Function": mass_func,
        "Type1": type1,
        "Type2": type2,
        "Detection Method": method,
        "Reference": reference,
        "Notes": notes,
    }
    return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

# ---------- Helper parsers ----------
def pm_to_triplet(value_str):
    """
    Convert 'value±err' strings into [err-, value, err+] triplets.
    Handles forms like '0.0+0.05' (upper-only error) as [0.0, 0.0, 0.05].
    Returns None when value is missing ('/').
    """
    s = value_str.strip()
    if s == "/" or s == "" or s.lower() == "nan":
        return None
    s = (
        s.replace("$", "")
         .replace("\\pm", "±")
         .replace("\\,", "")
         .replace("~", " ")
         .replace("\\", "")
    )
    # upper-only uncertainty like "0.0+0.05"
    if "+" in s and "±" not in s:
        parts = s.split("+")
        try:
            val = float(parts[0])
            errp = float(parts[1])
            return [0.0, val, errp]
        except Exception:
            pass
    # standard "value±err"
    if "±" in s:
        val_str, err_str = s.split("±")
        val = float(val_str)
        err = float(err_str)
        return [err, val, err]
    # plain number
    try:
        val = float(s)
        return [0.0, val, 0.0]
    except Exception:
        return None

def as_lower_limit_triplet(value):
    """Encode a lower limit as [0.0, value, +inf]."""
    if value is None:
        return None
    return [0.0, float(value), float("inf")]

# Parse 3 LaTeX tables into your observation schema, assemble a DataFrame,
# ---------- Table 1 rows (Period, Eccentricity, ref keys) ----------
tab1_rows = [
(1, "89 Her", "289.1±0.2", "0.29±0.07", "1"),
(2, "AC Her", "1188.9±1.2", "0.0+0.05", "2"),
(3, "BD+39 4926", "871.7±0.4", "0.024±0.006", "3"),
(4, "BD+46 442", "140.82±0.02", "0.085±0.005", "4"),
(5, "DY Ori", "1248±36", "0.22±0.08", "5"),
(6, "EP Lyr", "1151±14", "0.39±0.09", "5"),
(7, "HD 44179", "317.6±1.1", "0.27±0.03", "6,7"),
(8, "HD 46703", "597.4±0.2", "0.30±0.02", "8"),
(9, "HD 52961", "1288.6±0.3", "0.23±0.01", "7"),
(10, "HD 95767", "1989±61", "0.25±0.05", "9"),
(11, "HD 108015", "906.3±5.9", "0.0+0.03", "9"),
(12, "HD 131356", "1488.0±8.7", "0.32±0.04", "9"),
(13, "HD 158616", "363.3±1.0", "0.0+0.1", "10"),
(14, "HD 213985", "259.6±0.7", "0.21±0.05", "9"),
(15, "HP Lyr", "1818±80", "0.20±0.04", "5"),
(16, "HR 4049", "430.6±0.1", "0.30±0.01", "11"),
(17, "IRAS 05208-2035", "234.38±0.04", "0.0+0.02", "17"),
(18, "IRAS 06165+3158", "262.6±0.7", "0.0+0.05", ""),
(19, "IRAS 06452-3456", "215.4±0.4", "0.0+0.03", ""),
(20, "IRAS 08544-4431", "501.1±1.0", "0.20±0.02", "12"),
(21, "IRAS 09144-4933", "1762±27", "0.30±0.04", "5"),
(22, "IRAS 15469-5311", "390.2±0.7", "0.08±0.02", "12"),
(23, "IRAS 16230-3410", "649.8±3.5", "0.0+0.13", ""),
(24, "IRAS 17038-4815", "1394±12", "0.63±0.06", "5"),
(25, "IRAS 19125+0343", "519.7±0.7", "0.24±0.03", "12"),
(26, "IRAS 19135+3937", "126.97±0.08", "0.13±0.03", "13"),
(27, "IRAS 19157-0247", "119.6±0.1", "0.34±0.04", "12"),
(28, "RU Cen", "1489±10", "0.62±0.07", "14"),
(29, "SAO 173329", "115.951±0.002", "0.0+0.04", "9"),
(30, "ST Pup", "406.0±2.2", "0.0+0.04", "15"),
(31, "SX Cen", "564.3±7.6", "0.0+0.06", "14"),
(32, "TW Cam", "662.2±5.3", "0.25±0.04", "5"),
(33, "U Mon", "2550±143", "0.25±0.06", "16"),
]
tab1 = {name: {"Period": pm_to_triplet(per), "Eccentricity": pm_to_triplet(ecc), "RefKeys": ref}
        for _, name, per, ecc, ref in tab1_rows}

# ---------- Table 2 rows (a1 sin i, mass function, M2_min) ----------
tab2_rows = [
(1,"89 Her","0.106±0.007","0.0019±0.0004","0.10"),
(2,"AC Her","1.176±0.080","0.153±0.032","0.64"),
(3,"BD+39 4926","1.286±0.007","0.373±0.006","1.03"),
(4,"BD+46 442","0.3074±0.0014","0.195±0.003","0.72"),
(5,"DY Ori","1.39±0.11","0.23±0.05","0.79"),
(6,"EP Lyr","1.30±0.12","0.22±0.06","0.77"),
(7,"HD 44179","0.342±0.008","0.053±0.003","0.38"),
(8,"HD 46703","0.839±0.015","0.220±0.012","0.77"),
(9,"HD 52961","1.507±0.034","0.274±0.019","0.87"),
(10,"HD 95767","2.14±0.16","0.33±0.07","0.96"),
(11,"HD 108015","0.28±0.02","0.0036±0.0009","0.13"),
(12,"HD 131356","2.11±0.09","0.57±0.07","1.33"),
(13,"HD 158616","0.28±0.03","0.022±0.008","0.26"),
(14,"HD 213985","0.733±0.025","0.777±0.079","1.62"),
(15,"HP Lyr","1.27±0.06","0.083±0.007","0.47"),
(16,"HR 4049","0.627±0.010","0.177±0.008","0.69"),
(17,"IRAS 05208-2035","0.396±0.004","0.150±0.005","0.64"),
(18,"IRAS 06165+3158","0.374±0.011","0.10±0.01","0.52"),
(19,"IRAS 06452-3456","0.73±0.01","1.12±0.05","2.07"),
(20,"IRAS 08544-4431","0.398±0.008","0.033±0.002","0.31"),
(21,"IRAS 09144-4933","2.25±0.11","0.49±0.07","1.21"),
(22,"IRAS 15469-5311","0.438±0.015","0.074±0.008","0.45"),
(23,"IRAS 16230-3410","0.232±0.021","0.004±0.001","0.13"),
(24,"IRAS 17038-4815","1.52±0.08","0.24±0.04","0.81"),
(25,"IRAS 19125+0343","0.56±0.02","0.086±0.010","0.48"),
(26,"IRAS 19135+3937","0.209±0.008","0.075±0.008","0.45"),
(27,"IRAS 19157-0247","0.083±0.003","0.0053±0.0007","0.15"),
(28,"RU Cen","2.38±0.15","0.81±0.17","1.66"),
(29,"SAO 173329","0.132±0.003","0.023±0.001","0.27"),
(30,"ST Pup","0.67±0.02","0.241±0.026","0.81"),
(31,"SX Cen","1.12±0.05","0.58±0.07","1.35"),
(32,"TW Cam","0.83±0.04","0.174±0.022","0.68"),
(33,"U Mon","3.38±0.31","0.79±0.18","1.64"),
]
tab2 = {name: {"a1sini": pm_to_triplet(a1), "Mass Function": pm_to_triplet(f), "M2_min": float(m2min)}
        for _, name, a1, f, m2min in tab2_rows}

# ---------- Table 3 rows (for notes: E(B-V), L_IR/L_*, Depletion) ----------
tab3_rows = [
(1,"89 Her","6600","0.8","0.02","0.38","-0.5","mild"),
(2,"AC Her","5800","1.0","0.46","0.24","-1.4","mild"),
(3,"BD+39 4926","7750","1.0","0.23","0.0","-2.4","strong"),
(4,"BD+46 442","6250","1.5","0.23","0.19","-0.8","no"),
(5,"DY Ori","5900","1.5","0.90","0.74","-2.3","strong"),
(6,"EP Lyr","6200","1.5","0.48","0.04","-1.8","moderate"),
(7,"HD 44179","7500","0.8","0.15","18.1","-3.3","strong"),
(8,"HD 46703","6250","1.0","0.23","0.02","-1.7","mild"),
(9,"HD 52961","6000","0.5","0.04","0.13","-4.8","strong"),
(10,"HD 95767","7500","2.0","0.58","0.55","0.1","no"),
(11,"HD 108015","7000","1.5","0.15","1.04","-0.1","no"),
(12,"HD 131356","6000","1.0","0.15","0.65","0.0","mild"),
(13,"HD 158616","7250","1.25","0.51","0.23","-0.6","no"),
(14,"HD 213985","8250","1.5","0.12","0.35","-0.9","strong"),
(15,"HP Lyr","6300","1.0","0.39","0.56","-1.0","strong"),
(16,"HR 4049","7600","1.1","0.20","0.12","-4.8","strong"),
(17,"IRAS 05208-2035","4250","0.75","0.01","0.43","-0.7","no"),
(18,"IRAS 06165+3158","4250","1.5","0.53","0.39","-0.9","no"),
(19,"IRAS 06452-3456","/","/","0.94","0.11","/","/"),
(20,"IRAS 08544-4431","7250","1.5","1.32","0.49","-0.3","mild"),
(21,"IRAS 09144-4933","5750","0.5","1.78","0.81","-0.3","moderate"),
(22,"IRAS 15469-5311","7500","1.5","1.27","0.74","0.0","strong"),
(23,"IRAS 16230-3410","6250","1.0","0.72","0.46","-0.7","moderate"),
(24,"IRAS 17038-4815","4750","0.5","0.57","0.79","-1.5","no"),
(25,"IRAS 19125+0343","7750","1.0","0.94","0.90","-0.3","strong"),
(26,"IRAS 19135+3937","6000","0.5","0.28","0.26","-1.0","no"),
(27,"IRAS 19157-0247","7750","1.0","0.66","0.79","0.1","no"),
(28,"RU Cen","6000","1.5","0.18","0.39","-1.9","moderate"),
(29,"SAO 173329","7000","1.0","0.31","0.35","-0.9","no"),
(30,"ST Pup","5500","1.0","0.06","1.32","-1.5","strong"),
(31,"SX Cen","6250","1.5","0.17","0.40","-1.1","strong"),
(32,"TW Cam","4800","0.0","0.42","0.43","-0.5","no"),
(33,"U Mon","5000","0.0","0.34","0.28","-0.8","no"),
]
tab3 = {
    name: {
        "EBV": None if ebv == "/" else float(ebv),
        "LIR": None if lir == "/" else float(lir),
        "Depletion": depl,
    }
    for _, name, *_rest, ebv, lir, _feh, depl in tab3_rows
}


In [3]:
# # ---------- Build the DataFrame ----------
# for name in [r[1] for r in tab1_rows]:
#     per = tab1[name]["Period"]
#     ecc = tab1[name]["Eccentricity"]
#     refs = tab1[name]["RefKeys"]
#     mass_func = tab2[name]["Mass Function"]

#     notes_bits = []
#     if name in tab3:
#         ebv = tab3[name]["EBV"]
#         lir = tab3[name]["LIR"]
#         depl = tab3[name]["Depletion"]
#         if ebv is not None:
#             notes_bits.append(f"E(B-V)={ebv}")
#         if lir is not None:
#             notes_bits.append(f"L_IR/L_*={lir}")
#         if depl and depl != "/":
#             notes_bits.append(f"Depletion={depl}")
#     notes_bits.append("Companion minimum mass assumes M1=0.6 Msun and i=75 deg (from Table 2).")
#     notes = "; ".join(notes_bits)

#     # Add row (unknowns left None; M2 is a lower limit)
#     df = add_observation(
#         df,
#         system_name=name,
#         ra=None, dec=None,
#         period=per,
#         ecc=ecc,
#         m1=None,
#         m1_sin3i=None,
#         m2=as_lower_limit_triplet(tab2[name]["M2_min"]),
#         m2_sin3i=None,
#         q=None,
#         mass_func=mass_func,
#         type1=None, type2=None,
#         method=["RV"],
#         reference=[rk.strip() for rk in refs.split(",")] if refs else [],
#         notes=notes
#     )


TRI_NAN = [np.nan, np.nan, np.nan]
REF_ALL = ['2018A&A...620A..85O']   # single paper for all entries

def safe_triplet(x):
    """Return a [f,f,f] list; NaN triplet if x is None; cast floats otherwise."""
    if x is None:
        return TRI_NAN.copy()
    # assume iterable of length 3 from pm_to_triplet
    try:
        if len(x) == 3:
            return [float(x[0]), float(x[1]), float(x[2])]
    except Exception:
        pass
    # scalar → unknown errs
    try:
        v = float(x)
        return [np.nan, v, np.nan]
    except Exception:
        return TRI_NAN.copy()

def as_lower_limit_triplet(value):
    """Encode lower limit as [0, value, +inf]."""
    return [0.0, float(value), float('inf')] if value is not None else TRI_NAN.copy()

# ---------- Build the DataFrame (triplet-safe) ----------
for name in [r[1] for r in tab1_rows]:
    per  = safe_triplet(tab1[name]["Period"])
    ecc  = safe_triplet(tab1[name]["Eccentricity"])
    mf   = safe_triplet(tab2[name]["Mass Function"])
    m2ll = as_lower_limit_triplet(tab2[name]["M2_min"])

    # notes
    notes_bits = []
    if name in tab3:
        ebv  = tab3[name]["EBV"]
        lir  = tab3[name]["LIR"]
        depl = tab3[name]["Depletion"]
        if ebv is not None: notes_bits.append(f"E(B-V)={ebv}")
        if lir is not None: notes_bits.append(f"L_IR/L_*={lir}")
        if depl and depl != "/": notes_bits.append(f"Depletion={depl}")
    notes_bits.append("Companion minimum mass assumes M1=0.6 Msun and i=75 deg (from Table 2).")
    notes = "; ".join(notes_bits)

    df = add_observation(
        df,
        system_name=name,
        ra=TRI_NAN.copy(),                 # placeholder, to be overwritten by SIMBAD results
        dec=TRI_NAN.copy(),                # placeholder
        period=per,
        ecc=ecc,
        m1=TRI_NAN.copy(),                 # unknowns stored as NaN triplets (keeps dtype numeric)
        m1_sin3i=TRI_NAN.copy(),
        m2=m2ll,                           # lower-limit triplet
        m2_sin3i=TRI_NAN.copy(),
        q=TRI_NAN.copy(),
        mass_func=mf,
        type1=None, type2="post AGB",
        method=["RV"],
        reference=REF_ALL,                 # same ADS bibcode for all rows
        notes=notes
    )


## RA and DEC in the right format

In [4]:
import subprocess, re, ast
import sys

# Collect the system names from the DataFrame
targets = df["System Name"].tolist()

display(df['System Name'].values)
targets = df['System Name'].values.tolist()

# Run the script and capture output
result = subprocess.run(["python3", "Get_Coords_From_SIMBAD.py"] + targets, capture_output=True, text=True)


# print("STDOUT:\n", result.stdout)
# Script prints one dict per system like:
#  *__89_HER = {
#     "System Name": '*  89 Her',
#     "RA":  [0.00001144, 268.854951, 0.00001144],
#     "Dec": [0.00001275, 26.049991, 0.00001275],
# }

# Get the printed output as a string
RA_DEC = result.stdout

entries = re.findall(r'([A-Z0-9_]+)\s*=\s*({.*?})', RA_DEC, re.DOTALL)
RADEC_data = {}

for name, dict_str in entries:
    RADEC_data[name] = ast.literal_eval(dict_str)

RADEC_data.keys()
# Now `RADEC_data` is a Python dictionary with the object data
# print(RADEC_data['HD__58978']['RA'])
# print(RADEC_data['HD__58978']['Dec'])

# Extract RA and Dec values
RA_values = [RADEC_data[key]['RA'] for key in RADEC_data.keys() ]
DEC_values = [RADEC_data[key]['Dec'] for key in RADEC_data.keys() ]


print(RA_values)
print(DEC_values)

df["RA"]  = RA_values
df["Dec"] = DEC_values   


array(['89 Her', 'AC Her', 'BD+39 4926', 'BD+46 442', 'DY Ori', 'EP Lyr',
       'HD 44179', 'HD 46703', 'HD 52961', 'HD 95767', 'HD 108015',
       'HD 131356', 'HD 158616', 'HD 213985', 'HP Lyr', 'HR 4049',
       'IRAS 05208-2035', 'IRAS 06165+3158', 'IRAS 06452-3456',
       'IRAS 08544-4431', 'IRAS 09144-4933', 'IRAS 15469-5311',
       'IRAS 16230-3410', 'IRAS 17038-4815', 'IRAS 19125+0343',
       'IRAS 19135+3937', 'IRAS 19157-0247', 'RU Cen', 'SAO 173329',
       'ST Pup', 'SX Cen', 'TW Cam', 'U Mon'], dtype=object)

[[1.144e-05, 268.854951, 1.144e-05], [4.13e-06, 277.567655, 4.13e-06], [8.03e-06, 341.54678, 8.03e-06], [6.86e-06, 26.445963, 6.86e-06], [2.203e-05, 91.562119, 2.203e-05], [4.52e-06, 289.581488, 4.52e-06], [0.00050026, 94.992577, 0.00050026], [1.018e-05, 99.468445, 1.018e-05], [7.49e-06, 105.915127, 7.49e-06], [7.5e-06, 165.517981, 7.5e-06], [8.74e-06, 186.222932, 8.74e-06], [1.801e-05, 224.252855, 1.801e-05], [3.83e-06, 262.69552, 3.83e-06], [1.481e-05, 338.864692, 1.481e-05], [3.41e-06, 290.412781, 3.41e-06], [2.128e-05, 154.531625, 2.128e-05], [5.1e-06, 80.747595, 5.1e-06], [8.09e-06, 94.960932, 8.09e-06], [8.75e-06, 101.754738, 8.75e-06], [1.693e-05, 134.059083, 1.693e-05], [8.99e-06, 139.040148, 8.99e-06], [9.4e-06, 237.682521, 9.4e-06], [5.45e-06, 246.584932, 5.45e-06], [8.15e-06, 256.902745, 8.15e-06], [5.15e-06, 288.754917, 5.15e-06], [3.57e-06, 288.800572, 3.57e-06], [4.89e-06, 289.594652, 4.89e-06], [2.189e-05, 182.349229, 2.189e-05], [1.88e-06, 109.034479, 1.88e-06], [6.46e-

## Fix the references into a list of strings

In [5]:
# We are only referencing the Oomen paper, replace Reference column for all systems
df["Reference"] = [['2018A&A...620A..85O']] * len(df)


# Check what it looks like: 

In [6]:
display(df)

,System Name,RA,Dec,Period,Eccentricity,M1,M1_sin3i,M2,M2_sin3i,q,Mass Function,Type1,Type2,Detection Method,Reference,Notes
0,89 Her,"[1.144e-05, 268.854951, 1.144e-05]","[1.275e-05, 26.049991, 1.275e-05]","[0.2, 289.1, 0.2]","[0.07, 0.29, 0.07]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.1, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.0004, 0.0019, 0.0004]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.02; L_IR/L_*=0.38; Depletion=mild; Co...
1,AC Her,"[4.13e-06, 277.567655, 4.13e-06]","[7.5e-06, 21.866833, 7.5e-06]","[1.2, 1188.9, 1.2]","[0.0, 0.0, 0.05]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.64, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.032, 0.153, 0.032]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.46; L_IR/L_*=0.24; Depletion=mild; Co...
2,BD+39 4926,"[8.03e-06, 341.54678, 8.03e-06]","[6.53e-06, 40.107308, 6.53e-06]","[0.4, 871.7, 0.4]","[0.006, 0.024, 0.006]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 1.03, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.006, 0.373, 0.006]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.0; Depletion=strong; C...
3,BD+46 442,"[6.86e-06, 26.445963, 6.86e-06]","[3.42e-06, 46.816927, 3.42e-06]","[0.02, 140.82, 0.02]","[0.005, 0.085, 0.005]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.72, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.195, 0.003]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.19; Depletion=no; Comp...
4,DY Ori,"[2.203e-05, 91.562119, 2.203e-05]","[1.956e-05, 13.905311, 1.956e-05]","[36.0, 1248.0, 36.0]","[0.08, 0.22, 0.08]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.79, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.05, 0.23, 0.05]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.9; L_IR/L_*=0.74; Depletion=strong; C...
5,EP Lyr,"[4.52e-06, 289.581488, 4.52e-06]","[5.19e-06, 27.850856, 5.19e-06]","[14.0, 1151.0, 14.0]","[0.09, 0.39, 0.09]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.06, 0.22, 0.06]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.48; L_IR/L_*=0.04; Depletion=moderate...
6,HD 44179,"[0.00050026, 94.992577, 0.00050026]","[0.00040556, -10.637418, 0.00040556]","[1.1, 317.6, 1.1]","[0.03, 0.27, 0.03]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.38, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.003, 0.053, 0.003]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.15; L_IR/L_*=18.1; Depletion=strong; ...
7,HD 46703,"[1.018e-05, 99.468445, 1.018e-05]","[5.78e-06, 53.517211, 5.78e-06]","[0.2, 597.4, 0.2]","[0.02, 0.3, 0.02]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.77, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.012, 0.22, 0.012]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.23; L_IR/L_*=0.02; Depletion=mild; Co...
8,HD 52961,"[7.49e-06, 105.915127, 7.49e-06]","[6.44e-06, 10.770297, 6.44e-06]","[0.3, 1288.6, 0.3]","[0.01, 0.23, 0.01]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.87, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.019, 0.274, 0.019]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.04; L_IR/L_*=0.13; Depletion=strong; ...
9,HD 95767,"[7.5e-06, 165.517981, 7.5e-06]","[3.72e-06, -62.161898, 3.72e-06]","[61.0, 1989.0, 61.0]","[0.05, 0.25, 0.05]","[nan, nan, nan]","[nan, nan, nan]","[0.0, 0.96, inf]","[nan, nan, nan]","[nan, nan, nan]","[0.07, 0.33, 0.07]",None,post AGB,[RV],[2018A&A...620A..85O],E(B-V)=0.58; L_IR/L_*=0.55; Depletion=no; Comp...


## Finally convert and save your table to hdf5 format

In [7]:
# Extend the HDF5 saving function to include metadata
def save_triplet_columns_and_metadata(df, triplet_columns, h5_filename):
    with h5py.File(h5_filename, "w") as f:
        # Save each triplet column as a dataset
        for col in triplet_columns:
            try:
                arr = np.vstack(df[col].values)  # Shape (N, 3)
                f.create_dataset(col, data=arr)
            except Exception as e:
                print(f"Skipping column '{col}' due to: {e}")
        
        # Save all other columns as metadata in JSON format
        metadata_cols = [col for col in df.columns if col not in triplet_columns]
        metadata_df = df[metadata_cols]

        # Convert to JSON-serializable format and store as a string
        meta_json = metadata_df.to_json(orient="records")
        f.create_dataset("metadata_json", data=np.bytes_(meta_json))

#


In [8]:
#  Run the function with metadata saving
triplet_cols = ["RA", "Dec", "Period", "Eccentricity", "M1", "M1_sin3i", "M2", "M2_sin3i", "q", "Mass Function",]

from paths import DATA_DIR

save_triplet_columns_and_metadata(df, triplet_cols, DATA_DIR / "result_tables/post_AGB_stars.h5")
